In [1]:
layers = 28
kv_heads = 8
head_dim = 128
bytes_per_elem = 2  # fp16

kv_bytes_per_token = 2 * layers * kv_heads * head_dim * bytes_per_elem
print(f"KV bytes/token = {kv_bytes_per_token} bytes = {kv_bytes_per_token/1024:.1f} KiB")

gpu_mem = 24e9
util = 0.92
weights = 4.2e9 * 2
overhead = 1.6e9

usable = gpu_mem * util
kv_budget = usable - weights - overhead
max_tokens = kv_budget / kv_bytes_per_token
max_seqs = max_tokens / 4096

print(f"usable memory = {usable/1e9:.2f} GB")
print(f"weights = {weights/1e9:.2f} GB")
print(f"KV budget = {kv_budget/1e9:.2f} GB")
print(f"max cacheable tokens = {max_tokens:,.0f}")
print(f"max concurrent 4096-seqs = {max_seqs:.1f}")

KV bytes/token = 114688 bytes = 112.0 KiB
usable memory = 22.08 GB
weights = 8.40 GB
KV budget = 12.08 GB
max cacheable tokens = 105,329
max concurrent 4096-seqs = 25.7


In [5]:
import csv

log_path = "bench_log.csv"  # now in the same folder as the notebook

rows = []
with open(log_path) as f:
    for row in csv.DictReader(f):
        rows.append({k: float(v) if "." in v else int(v) for k, v in row.items()})

print(f"{'batch':>6}{'tokens_in_flight':>18}{'predicted_util':>16}{'logged_util':>14}")
for r in rows:
    if r["prompt_len"] == 3584:
        tokens_in_flight = r["batch_size"] * (r["prompt_len"] + r["gen_len"])
        predicted = tokens_in_flight / max_tokens
        print(f"{r['batch_size']:>6}{tokens_in_flight:>18,}{predicted:>16.3f}{r['kv_cache_util']:>14}")

 batch  tokens_in_flight  predicted_util   logged_util
     4            16,384           0.156          0.16
     8            32,768           0.311          0.31
    16            65,536           0.622          0.62
    24            98,304           0.933          0.93
    32           131,072           1.244          0.97
    48           196,608           1.867          0.97


In [6]:
print(f"{'batch':>6}{'reported':>11}{'(p+g)*n/wall':>15}{'goodput(gen only)':>20}{'preempted':>11}")
for r in rows:
    if r["prompt_len"] == 3584:
        predicted = (r["prompt_len"]+r["gen_len"]) * r["num_requests"] / r["wall_clock_s"]
        goodput = r["gen_len"] * r["num_requests"] / r["wall_clock_s"]
        print(f"{r['batch_size']:>6}{r['reported_tok_s']:>11.1f}{predicted:>15.1f}"
              f"{goodput:>20.1f}{r['preempted_seqs']:>11}")

 batch   reported   (p+g)*n/wall   goodput(gen only)  preempted
     4      565.4          565.4                70.7          0
     8      902.6          902.7               112.8          0
    16     1311.4         1311.5               163.9          0
    24     1607.4         1607.3               200.9          0
    32     1384.0         1383.9               173.0          7
    48     1298.5         1298.5               162.3         23


In [7]:
r24 = next(r for r in rows if r["prompt_len"]==3584 and r["batch_size"]==24)
method2 = r24["num_requests"] * (1000 / r24["itl_ms_p50"])
print(f"Method 2 (batch * 1000/itl_ms_p50) = {method2:.1f} tok/s")

r48 = next(r for r in rows if r["prompt_len"]==3584 and r["batch_size"]==48)
real48_goodput = r48["gen_len"] * r48["num_requests"] / r48["wall_clock_s"]
print(f"Actual batch-48 goodput = {real48_goodput:.1f} tok/s (report predicted ~3200)")

Method 2 (batch * 1000/itl_ms_p50) = 249.8 tok/s
Actual batch-48 goodput = 162.3 tok/s (report predicted ~3200)
